Governance Demo

In [ ]:
USE ROLE ACCOUNTADMIN;
USE SECONDARY ROLES NONE;

USE DATABASE zFACETS_DEV_CLONE;
USE SCHEMA SILVER;
USE WAREHOUSE WH_XS;

-- Step 1a: Show the raw problem — PHI is fully exposed with no controls
SELECT
    MEME_ID,
    MEME_LAST_NAME,
    MEME_FIRST_NAME,
    MEME_DOB,
    MEME_SEX,
    MECD_AID_CD,       -- Medi-Cal aid code — HIPAA PHI
    MECD_BIC,          -- Medi-Cal beneficiary ID — HIPAA PHI
    MEME_MCTR_TYPE,
    ACTIVE_PCP_NAME
FROM MEMBER
ORDER BY MEME_ID
LIMIT 10;


In [ ]:
-- Step 1b: Show the DATA_CLASSIFICATION tags applied by setup
SELECT
    COLUMN_NAME,
    TAG_VALUE AS classification_level
FROM TABLE(
    INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS(
        'MEMBER', 'table'
    )
)
WHERE TAG_NAME = 'DATA_CLASSIFICATION'
ORDER BY
    CASE TAG_VALUE
        WHEN 'PII'        THEN 1
        WHEN 'RESTRICTED' THEN 2
        WHEN 'SENSITIVE'  THEN 3
        WHEN 'INTERNAL'   THEN 4
        WHEN 'PUBLIC'     THEN 5
    END,
    COLUMN_NAME;

In [ ]:
CREATE OR REPLACE TABLE MEMBER_COPY
    AS SELECT * FROM MEMBER;

SELECT
    COLUMN_NAME,
    TAG_VALUE AS classification_level
FROM TABLE(
    INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS(
        'MEMBER_COPY', 'table'
    )
)
WHERE TAG_NAME = 'DATA_CLASSIFICATION'
ORDER BY
    CASE TAG_VALUE
        WHEN 'PII' THEN 1 WHEN 'RESTRICTED' THEN 2
        WHEN 'SENSITIVE' THEN 3 WHEN 'INTERNAL' THEN 4
    END, COLUMN_NAME;

-- Role privilege matrix:
-- ┌──────────────────────────┬─────────────┬──────────────┬───────────────────┬──────────────────┐
-- │ Classification           │ ACCOUNTADMIN│ DATA_ENGINEER│ ANALYTICS_INNOVATOR│ BUSINESS_ANALYST │
-- ├──────────────────────────┼─────────────┼──────────────┼───────────────────┼──────────────────┤
-- │ PII   (MECD_AID_CD/BIC)  │ Full        │ Full         │ ***PHI REDACTED***│ ***PHI REDACTED**│
-- │ RESTRICTED (DOB)         │ Full        │ Full         │ Year only         │ NULL             │
-- │ SENSITIVE (name, sex)    │ Full        │ Full         │ First initial+*** │ ***SENSITIVE***  │
-- │ INTERNAL                 │ Full        │ Full         │ Full              │ Full             │
-- └──────────────────────────┴─────────────┴──────────────┴───────────────────┴──────────────────┘

In [ ]:
USE ROLE DATA_ENGINEER_ROLE;

SELECT
    MEME_ID,
    MEME_LAST_NAME,       -- SENSITIVE  → full | J****** | *** SENSITIVE ***
    MEME_FIRST_NAME,      -- SENSITIVE  → full | J****** | *** SENSITIVE ***
    MEME_DOB,             -- RESTRICTED → exact date | year only (HIPAA §164.514(b)) | NULL
    MEME_SEX,             -- SENSITIVE  → full | M* | *** SENSITIVE ***
    MECD_AID_CD,          -- PII        → full | *** PHI REDACTED *** | *** PHI REDACTED ***
    MECD_BIC,             -- PII        → full | *** PHI REDACTED *** | *** PHI REDACTED ***
    MEME_MCTR_TYPE,       -- INTERNAL   → visible for all roles
    ACTIVE_PCP_NAME       -- SENSITIVE  → full | D****** | *** SENSITIVE ***
FROM MEMBER
ORDER BY MEME_ID
LIMIT 10;

In [ ]:
%%sql -r dataframe_5
USE ROLE ANALYTICS_INNOVATOR_ROLE;

SELECT
    MEME_ID,
    MEME_LAST_NAME,       -- SENSITIVE  → full | J****** | *** SENSITIVE ***
    MEME_FIRST_NAME,      -- SENSITIVE  → full | J****** | *** SENSITIVE ***
    MEME_DOB,             -- RESTRICTED → exact date | year only (HIPAA §164.514(b)) | NULL
    MEME_SEX,             -- SENSITIVE  → full | M* | *** SENSITIVE ***
    MECD_AID_CD,          -- PII        → full | *** PHI REDACTED *** | *** PHI REDACTED ***
    MECD_BIC,             -- PII        → full | *** PHI REDACTED *** | *** PHI REDACTED ***
    MEME_MCTR_TYPE,       -- INTERNAL   → visible for all roles
    ACTIVE_PCP_NAME       -- SENSITIVE  → full | D****** | *** SENSITIVE ***
FROM MEMBER
ORDER BY MEME_ID
LIMIT 10;

In [ ]:
USE ROLE BUSINESS_ANALYST_ROLE;

SELECT
    MEME_ID,
    MEME_LAST_NAME,       -- SENSITIVE  → full | J****** | *** SENSITIVE ***
    MEME_FIRST_NAME,      -- SENSITIVE  → full | J****** | *** SENSITIVE ***
    MEME_DOB,             -- RESTRICTED → exact date | year only (HIPAA §164.514(b)) | NULL
    MEME_SEX,             -- SENSITIVE  → full | M* | *** SENSITIVE ***
    MECD_AID_CD,          -- PII        → full | *** PHI REDACTED *** | *** PHI REDACTED ***
    MECD_BIC,             -- PII        → full | *** PHI REDACTED *** | *** PHI REDACTED ***
    MEME_MCTR_TYPE,       -- INTERNAL   → visible for all roles
    ACTIVE_PCP_NAME       -- SENSITIVE  → full | D****** | *** SENSITIVE ***
FROM MEMBER
ORDER BY MEME_ID
LIMIT 10;

-- ┌─────────────────────────┬─────────────┬──────────────┬───────────────────┬──────────────────┐
-- │ Plan Type               │ ACCOUNTADMIN│ DATA_ENGINEER│ ANALYTICS_INNOVATOR│ BUSINESS_ANALYST │
-- ├─────────────────────────┼─────────────┼──────────────┼───────────────────┼──────────────────┤
-- │ COMM   (~30,593 rows)   │ ✓ Visible   │ ✓ Visible    │ ✓ Visible         │ ✓ Visible        │
-- │ DSNP   (~30,866 rows)   │ ✓ Visible   │ ✓ Visible    │ ✓ Visible         │ ✗ Hidden         │
-- │ MEDCAID (~30,642 rows)  │ ✓ Visible   │ ✓ Visible    │ ✗ Hidden          │ ✗ Hidden         │
-- └─────────────────────────┴─────────────┴──────────────┴───────────────────┴──────────────────┘

In [ ]:
USE ROLE DATA_ENGINEER_ROLE;

SELECT MEME_MCTR_TYPE, COUNT(*) AS member_count
FROM MEMBER
GROUP BY MEME_MCTR_TYPE
ORDER BY member_count DESC;

In [ ]:
USE ROLE ANALYTICS_INNOVATOR_ROLE;

SELECT MEME_MCTR_TYPE, COUNT(*) AS member_count
FROM MEMBER
GROUP BY MEME_MCTR_TYPE
ORDER BY member_count DESC;

In [ ]:
USE ROLE BUSINESS_ANALYST_ROLE;

SELECT MEME_MCTR_TYPE, COUNT(*) AS member_count
FROM MEMBER
GROUP BY MEME_MCTR_TYPE
ORDER BY member_count DESC;

In [ ]:
USE ROLE BUSINESS_ANALYST_ROLE;
SELECT *
FROM MEMBER
WHERE MEME_MCTR_TYPE = 'DSNP'
LIMIT 5;